# Lecture 02 — Advanced: Glossary LoRA Fine-tuning

Train on course glossary Q&A and compare the same evaluation questions before and after training.

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 3,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/aie2026_advanced",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## Load the training and evaluation datasets

In [ ]:

from pathlib import Path
import json
from google.colab import files

required = ["glossary_sft.jsonl", "advanced_eval.json"]

if any(not Path("/content", name).exists() for name in required):
    print("Upload glossary_sft.jsonl and advanced_eval.json together.")
    files.upload()

glossary_records = [
    json.loads(line)
    for line in Path("/content/glossary_sft.jsonl").read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

evaluation = json.loads(
    Path("/content/advanced_eval.json").read_text(encoding="utf-8")
)

known_terms = sorted({
    row["messages"][0]["content"][len("What is "):-1]
    for row in glossary_records
    if row["messages"][0]["content"].startswith("What is ")
})

training_questions = {
    row["messages"][0]["content"] for row in glossary_records
}

assert len(glossary_records) == 130
assert len(known_terms) == 65
assert len(evaluation) == 15
assert all(q["question"] not in training_questions for q in evaluation)

for category in ("grounded", "paraphrased", "ungrounded"):
    assert sum(q["category"] == category for q in evaluation) == 5

print("Training examples:", len(glossary_records))
print("Glossary terms:", len(known_terms))
print("Evaluation questions:", len(evaluation))
print("Dataset validation passed.")


## Load the original base model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)


SYSTEM = (
    "You are a study assistant for a fixed course glossary. "
    "These are the glossary entry names, without their definitions: "
    + ", ".join(known_terms)
    + ". If the requested term is not on this list, respond exactly: "
    "'Not covered in the course glossary.' "
    "For listed terms, explain the meaning clearly and briefly. "
    "Do not invent course-specific details."
)

def generate(model, question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            pad_token_id=tok.pad_token_id,
            **CONFIG["gen"],
        )
    return tok.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()


## Evaluate before fine-tuning

In [ ]:

import pandas as pd

def evaluate_glossary(model, label):
    rows = []
    start = time.time()

    for item in evaluation:
        response = generate(model, item["question"])
        refused = "not covered in the course glossary" in response.lower()

        rows.append({
            **item,
            "response": response,
            "refused": refused,
            "manual_correct": None,
        })

    known = [
        r for r in rows if r["category"] != "ungrounded"
    ]
    unknown = [
        r for r in rows if r["category"] == "ungrounded"
    ]

    summary = {
        "label": label,
        "n": len(rows),
        "known_non_refusal_rate": sum(
            not r["refused"] for r in known
        ) / len(known),
        "unknown_review_rate": sum(
            r["refused"] for r in unknown
        ) / len(unknown),
        "seconds": round(time.time() - start, 1),
    }

    (WORK / f"advanced_eval_{label}.json").write_text(
        json.dumps(
            {"summary": summary, "rows": rows},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(json.dumps(summary, indent=2))
    return summary, rows


In [ ]:
before_summary, before_rows = evaluate_glossary(model, "before")

## Train a fresh LoRA adapter on the glossary

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(glossary_records)

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
train_stats = trainer.train()  # 途中で止まった場合の再開は 4b を参照
training_seconds = time.time() - t0
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

## Compare results and inspect the answers

In [ ]:

after_summary, after_rows = evaluate_glossary(tuned, "after")

print("\nComparison:")
display(pd.DataFrame([before_summary, after_summary]))

print("\nReview the answers against the gold definitions:")
comparison = pd.DataFrame([
    {
        "ID": b["id"],
        "Category": b["category"],
        "Question": b["question"],
        "Expected": b["gold_answer"],
        "Before": b["response"],
        "After": a["response"],
    }
    for b, a in zip(before_rows, after_rows)
])
pd.set_option("display.max_colwidth", None)
display(comparison)

result = {
    "config": {
        k: v for k, v in CONFIG.items()
        if k not in ("work_dir", "drive_dir")
    },
    "training_examples": len(glossary_records),
    "glossary_terms": len(known_terms),
    "evaluation_questions": len(evaluation),
    "training_steps": trainer.state.global_step,
    "training_seconds": round(training_seconds, 1),
    "training_loss": train_stats.metrics.get("train_loss"),
    "before": before_summary,
    "after": after_summary,
    "executed_training_myself": True,
    "note": (
        "Known-term responses require manual semantic review. "
        "The automated rates measure refusal behavior, not "
        "definition accuracy."
    ),
}

(WORK / "advanced_result.json").write_text(
    json.dumps(result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("\nSaved advanced_result.json.")


## Download the results

In [ ]:

from google.colab import files
from zipfile import ZipFile

archive_path = Path("/content/lecture02_advanced_results.zip")
names = [
    "advanced_result.json",
    "advanced_eval_before.json",
    "advanced_eval_after.json",
]

with ZipFile(archive_path, "w") as archive:
    for name in names:
        file_path = WORK / name
        assert file_path.exists(), f"Missing {file_path}"
        archive.write(file_path, arcname=name)

files.download(str(archive_path))
